# Retrieval Augmented Generation (RAG)

This notebook wires the vectorstore pipeline (chunk → embed → similarity search → rerank)
to Claude via a LangChain LCEL chain, producing a working end-to-end RAG system.

1. LCEL Concepts: What a `Runnable` is and why LCEL is composable.
2. Load Vector Store: Load from disk (no re-embedding).
3. Format a Prompt
4. Build a RAG Chain: `build_rag_chain(vectorstore)`.
5. Run a Query: Invoke the chain, print the answer.
6. Inspect Retrieved Context: Show the reranked abstracts that informed the answer.
7. Out-of-Context Query: Check LLMs ability to admit when it doesn't know.
8. Critic Model + Refinement Loop: `invoke_and_review` with self-correction.
9. More Queries

In [1]:
import logging

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO)

# Vectorstore files were persisted here in notebook 1.vectorstore.ipynb
PERSIST_DIR = ".data/vectorstore"
COLLECTION_NAME = "arabidopsis_abstracts"
RAG_MODEL = "claude-sonnet-4-6"

### 1. LCEL Concepts

**LCEL** (LangChain Expression Language) is a declarative way to compose `Runnable`
objects into pipelines using the `|` operator — similar to Unix pipes.

Every component in the chain implements the same `Runnable` interface:
- `invoke(input)` — run it on a single input
- `stream(input)` — stream output tokens as they arrive
- `batch(inputs)` — run in parallel over a list of inputs

This uniformity is why you can compose them freely:
```python
chain = retriever | prompt | llm | output_parser
chain.invoke("my question")  # works exactly like calling any single Runnable
```

Our RAG chain shape:
```
query (str)
  ↓
{context: retrieve_and_rerank | format_context, question: passthrough}
  ↓
ChatPromptTemplate  →  ChatAnthropic (structured output)
  ↓
RagResult(answer, references)
```

### 2. Load Vector Store

Load the Chroma store built in `1.vectorstore.ipynb` — no re-embedding needed.

In [2]:
from llm_knowledge_discovery.vectorstore import load_vectorstore

vectorstore = load_vectorstore(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"Vector store loaded from: {PERSIST_DIR}")
print(f"Collection: {COLLECTION_NAME}")

/Users/dylanelliott/workspace/llm-knowledge-discovery/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Loading vectorstore: collection='arabidopsis_abstracts', persist_dir='.data/vectorstore'
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:llm_knowledge_discovery.vectorstore.store:[store.py] Vectorstore loaded from '.data/vectorstore'


Vector store loaded from: .data/vectorstore
Collection: arabidopsis_abstracts


### 3. Format a Prompt

The `RAG_PROMPT_TEMPLATE` defines the exact messages sent to Claude.
We can format it with sample inputs to see what the model receives.

In [3]:
from llm_knowledge_discovery.rag.chain import RAG_PROMPT_TEMPLATE

# Format with a sample input to see exactly what Claude will receive
sample = RAG_PROMPT_TEMPLATE.format_messages(
    context="Title: Flowering time in Arabidopsis\n\nFT gene promotes flowering under long days.",
    question="What genes regulate flowering time?",
)

print("Formatted prompt (sample):")
for msg in sample:
    print(f"\n[{msg.type}]\n{msg.content}")

Formatted prompt (sample):

[system]
You are a plant biology research assistant. Answer the user's question using ONLY the provided abstracts. If the abstracts do not contain enough information to answer the question, say so and do not include a references section. If you can answer the question, select only the abstracts you actually use, renumber them sequentially starting from [1], cite them inline by their new number (e.g. [1], [2]), and include a References section at the end listing only the abstracts you cited in that same sequential order, using their exact titles as they appear in the context — do not paraphrase or summarize titles.

[human]
Abstracts:

Title: Flowering time in Arabidopsis

FT gene promotes flowering under long days.

---

Question: What genes regulate flowering time?


### 4. Build the RAG Chain

`build_rag_chain` wires together three stages:
1. Retrieve + Rerank: Similarity search (bi-encoder, fast) → cross-encoder rerank (accurate)
2. Prompt: Injects the formatted abstracts + question into a `ChatPromptTemplate`
3. Generate: Sends the prompt to Claude and parses the response to a plain string

Defaults: `retrieval_k=10` (candidate pool), `rerank_k=5` (abstracts in the prompt).

In [4]:
from llm_knowledge_discovery.rag import build_rag_chain

chain = build_rag_chain(
    vectorstore=vectorstore,
    model=RAG_MODEL,
    retrieval_k=10,
    rerank_k=5
)

print(f"RAG Chain:\n{chain}")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] RAG chain built: model='claude-sonnet-4-6', retrieval_k=10, rerank_k=5


RAG Chain:
first={
  context: RunnableLambda(lambda q: retrieve_and_rerank(q, vectorstore, retrieval_k, rerank_k))
           | RunnableLambda(_format_context),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are a plant biology research assistant. Answer the user's question using ONLY the provided abstracts. If the abstracts do not contain enough information to answer the question, say so and do not include a references section. If you can answer the question, select only the abstracts you actually use, renumber them sequentially starting from [1], cite them inline by their new number (e.g. [1], [2]), and include a References section at the end listing only the abstracts you cited in that same sequential order, using their exact titles as they appear in the conte

### 5. Run a Query

Invoke the chain end-to-end. The chain:
1. Searches the vectorstore for the 10 most similar abstracts
2. Reranks them with the cross-encoder, keeping the top 5
3. Formats them into the prompt and sends to Claude
4. Returns Claude's answer as a plain string

In [5]:
query = "What genes regulate flowering time in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What genes regulate flowering time in Arabidopsis?

Answer:
Multiple genes and transcription factors have been identified as regulators of flowering time in Arabidopsis:

1. **CONSTANS (CO)** is a central gene in the photoperiodic pathway. Its expression is closely regulated by day length and modulated by both environmental and endogenous cues [1]. CO activates the expression of the florigen FLOWERING LOCUS T (FT) in leaves at the end of a long day [4]. CO expression is strongly regulated by the circadian clock, and CO itself feeds back to upregulate key circadian clock genes such as CCA1, LHY, PRR5, and GI by binding to their promoters [4].

2. **FLOWERING LOCUS T (FT)** is the florigen gene whose expression is a key output of multiple flowering pathways. Its expression is repressed by BZR1 [1], suppressed by IDD14 binding directly to its promoter [2], and regulated by TCP transcription factors [5].

3. **BRASSINAZOLE RESISTANT 1 (BZR1)**, a transcription factor in the brassino

### 6. Inspect Retrieved Context

To see *which* abstracts informed the answer, we call `retrieve_and_rerank` directly —
the same function the chain calls internally.

In [6]:
from llm_knowledge_discovery.rag import retrieve_and_rerank

reranked = retrieve_and_rerank(query, vectorstore)

print(f"Top {len(reranked)} abstracts that informed the answer:\n")
for i, doc in enumerate(reranked):
    print(f"[{i+1}] {doc.metadata['title']}")
    print(f"{doc.page_content[:200]}...")
    print()

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.77s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents


Top 5 abstracts that informed the answer:

[1] BRASSINAZOLE RESISTANT 1 delays photoperiodic flowering by repressing CONSTANS transcription.
Photoperiodic regulation of flowering time plays a critical role in plant reproductive success and crop yield. In Arabidopsis thaliana, the expression of the CONSTANS (CO) gene is closely regulated by...

[2] The Arabidopsis IDD14, IDD15, and IDD16 interact with DELLA proteins to negatively regulate flowering.
Flowering is a crucial developmental process in angiosperms. However, the underlying mechanisms remain to be elucidated. In this study, we identify INDETERMINATE DOMAIN (IDD) transcription factors (TF...

[3] Epigenetic regulation of floral transition: pathways and players.
The epigenetic mechanisms that regulate the DNA-histone contacts and the chromatin-based control of transcription provide an essential link between various signaling pathways in plants. The developmen...

[4] CONSTANS alters the circadian clock in Arabidopsis thaliana.
Pl

### 7. Out-of-Context Query

In [7]:
query = "What genes regulate unobtainium production in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What genes regulate unobtainium production in Arabidopsis?

Answer:
The provided abstracts do not contain any information about "unobtainium" or its production in Arabidopsis. "Unobtainium" is not a recognized biological compound or pathway in plant biology. Therefore, this question cannot be answered based on the available abstracts.

References:
[]



### 8. Critic Model + Refinement Loop

`invoke_and_review` wraps the full RAG pipeline with an optional self-correction loop:

1. **Retrieve + rerank** docs once (same docs used for both generation and critic checks)
2. **Generate** an initial answer
3. **Critique** via a second LLM call that checks two things only:
   - Citation integrity: every `[N]` inline has a matching References entry and vice versa
   - Title accuracy: every References title exactly matches the source document title (no paraphrasing)
4. If the critique **fails** and `review_steps > 1`, feed the issues back as a follow-up turn and regenerate
5. Repeat up to `review_steps` times, then return `(final_answer, list[CritiqueResult])`

**Why a plain Python loop instead of LangGraph?**
There's no branching complexity here — just generate → critique → (optionally) regenerate.
LangGraph would add graph definition overhead for what is fundamentally a `for` loop.
A plain loop is more readable, easier to debug, and has no checkpointing overhead.

In [8]:
from llm_knowledge_discovery.rag import invoke_and_review

# Use the flowering time query (same as Section 5) — multi-reference answer
# is a richer test for the critic than the out-of-context query.
# review_steps=2: generate → critique → (if failed) regenerate → critique again
query = "What genes regulate flowering time in Arabidopsis?"

final_rag_result, critiques = invoke_and_review(
    query=query,
    vectorstore=vectorstore,
    review_steps=2,
    model=RAG_MODEL
)

for i, c in enumerate(critiques, start=1):
    print(f"--- Critique {i} ---")
    print(f"Passed: {c.passed}")
    for issue in c.issues:
        print(f"  Issue: {issue}")
    if c.feedback:
        print(f"  Feedback: {c.feedback}")
    print()

print("Final Result:")
print(f"Query:\n{query}\n")
print(f"Answer:\n{final_rag_result.answer}\n")
print(f"References:\n{final_rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.rag.chain:[chain.py] Initial answer generated (2757 chars)
INFO:llm_knowledge_discovery.rag.chain:[chain.py] Critique step 1/2
INFO:llm_knowledge_discovery.rag.critic:[critic.py] Running critic on answer with 5 source docs
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.rag.critic:[critic.py] Critique result: passed=True, issues=

--- Critique 1 ---
Passed: True
  Feedback: All five inline citations [1]–[5] have matching entries in the References section. The References section is numbered sequentially from [1] to [5] with no gaps. Every title in the References section exactly matches one of the provided source document titles verbatim. No citation integrity, numbering, or title accuracy issues were found.

Final Result:
Query:
What genes regulate flowering time in Arabidopsis?

Answer:
Multiple genes and transcription factors regulate flowering time in Arabidopsis thaliana, working through several interconnected pathways:

**Key Floral Integrators:**
- **FLOWERING LOCUS T (FT)**, the florigen gene, is a central target of multiple flowering regulators. Its expression is activated or repressed by numerous upstream factors [1, 2, 4, 5].
- **CONSTANS (CO)** is a central activator of FT expression in leaves at the end of a long day, and its expression is tightly regulated by day length, environmental, and endogenous

### 9. More Queries

In [9]:
query = "What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?"

rag_result = chain.invoke(query)

print(f"Query:\n{query}\n")
print(f"Answer:\n{rag_result.answer}\n")
print(f"References:\n{rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Query:
What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?

Answer:
Based on the provided abstracts, two strategies in Arabidopsis can be highlighted as potentially relevant to seed size and weight:

1. **TCP4 knockout (loss-of-function):** Disruption of CIN-like TCP transcription factor function — specifically in the triple mutant *tcp3/4/10* — causes **enlarged seeds** due to delayed endosperm cellularization and accelerated seed coat growth [3]. TCP4 normally represses key pro-growth factors such as MINISEED3 (MINI3), SHORT HYPOCOTYL UNDER BLUE1 (SHB1), and AINTEGUMENTA (ANT). Therefore, single knockout or reduction of TCP4 function could be a strategy to increase seed size, as it would relieve repression of the MINI3-SHB1-IKU pathway and the ANT-driven integument/seed coat growth program [3].

2. **DOG1 knockout:** Large seeds in Arabidopsis have been shown to resemble *DOG1* knockout mutant seeds at the transc

In [10]:
query = "What sort of single knock-out or over-expression strategies could potentially increase seed size and weight in Arabidopsis?"

final_rag_result, critiques = invoke_and_review(
    query=query,
    vectorstore=vectorstore,
    review_steps=2,
    model=RAG_MODEL
)

for i, c in enumerate(critiques, start=1):
    print(f"--- Critique {i} ---")
    print(f"Passed: {c.passed}")
    for issue in c.issues:
        print(f"  Issue: {issue}")
    if c.feedback:
        print(f"  Feedback: {c.feedback}")
    print()

print("Final Result:")
print(f"Query:\n{query}\n")
print(f"Answer:\n{final_rag_result.answer}\n")
print(f"References:\n{final_rag_result.references}\n")

INFO:llm_knowledge_discovery.rag.chain:[chain.py] Retrieved 10 candidates, reranking to top 5
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking 10 documents with model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=5
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
INFO:llm_knowledge_discovery.vectorstore.reranking:[reranking.py] Reranking complete, returning top 5 documents
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.rag.chain:[chain.py] Initial answer generated (2038 chars)
INFO:llm_knowledge_discovery.rag.chain:[chain.py] Critique step 1/2
INFO:llm_knowledge_discovery.rag.critic:[critic.py] Running critic on answer with 5 source docs
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:llm_knowledge_discovery.rag.critic:[critic.py] Critique result: passed=False, issues

--- Critique 1 ---
Passed: False
  Issue: Citation integrity: Inline citation [3] appears multiple times in the answer body, but the References section only contains [1] and [2]. There is no [3] entry in the References section, making [3] an unmatched inline citation.
  Issue: Title accuracy: References [1] in the answer's References section is titled 'The Arabidopsis transcription factor TCP4 controls seed size by repressing the MINI3-SHB1-IKU pathway.' This title matches source document [3], not source document [1]. The numbering mismatch aside, the answer's References [1] label does not correspond to the inline [3] citations used in the body — the answer uses [3] inline but lists the TCP4 paper as References [1], creating a disconnect.
  Issue: Sequential numbering: The References section contains only [1] and [2], but inline citations include [3]. The number [3] is used inline but has no matching References entry, breaking sequential completeness.

--- Critique 2 ---
Passed: True
 